# Visualize CNN Feature Maps

Feature maps are the activations produced by convolution layers. They show which parts of an image activate each learned channel.

## Initialize PyTorch

- Import PyTorch, torchvision, and plotting tools
- Seed PyTorch for reproducibility
- Detect device (GPU, Apple Silicon, CPU)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

## Load the Dataset

- Load MNIST train and test data
- Convert images to tensors
- Use small subsets for fast training and visualization

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(4000)), batch_size=64, shuffle=True)
test_loader = DataLoader(Subset(test_data, range(200)), batch_size=1)

## Define CNN Model

- Build a CNN with named convolution layers
- Use named layers so hooks can capture feature maps
- Classify digits from learned convolution features

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return self.classifier(x)

model = MNISTCNN().to(device)

## Train CNN Briefly

- Train the model for one quick pass through the subset
- Create learned filters and activations for visualization
- Keep training short so the notebook stays interactive

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    loss = loss_fn(model(images), labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

## Register Activation Hooks

- Create a dictionary for saved feature maps
- Register forward hooks on convolution layers
- Store layer outputs whenever the model runs

In [ ]:
activations = {}

def save_activation(name):
    def hook(module, inputs, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = [
    model.conv1.register_forward_hook(save_activation("conv1")),
    model.conv2.register_forward_hook(save_activation("conv2")),
]

## Run One Image

- Select one test image
- Run it through the model to trigger hooks
- Display the input digit and predicted label

In [ ]:
model.eval()
image, label = next(iter(test_loader))

with torch.no_grad():
    prediction = model(image.to(device)).argmax(dim=1).item()

plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"input digit: true={label.item()}, pred={prediction}")
plt.axis("off");

## Plot Feature Maps

- Visualize activation maps from convolution layers
- Show how different channels respond to different image patterns
- Compare early and deeper feature maps

In [ ]:
def plot_feature_maps(feature_maps, title, limit=8):
    maps = feature_maps[0]
    n = min(limit, maps.shape[0])
    fig, axes = plt.subplots(1, n, figsize=(1.8 * n, 2))
    fig.suptitle(title)
    for ax, fmap in zip(axes, maps[:n]):
        ax.imshow(fmap, cmap="viridis")
        ax.axis("off")
    plt.tight_layout()

plot_feature_maps(activations["conv1"], "conv1 feature maps")
plot_feature_maps(activations["conv2"], "conv2 feature maps")

## Remove Hooks

- Remove registered hooks after collecting activations
- Keep the model from saving extra outputs in later cells

In [ ]:
for hook in hooks:
    hook.remove()